In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    ConfusionMatrixDisplay,
)

In [5]:
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

n_samples = 2000
annual_income = rng.normal(60000, 22000, n_samples).clip(15000, 200000)
credit_score = rng.normal(680, 75, n_samples).clip(300, 850)
loan_amount = rng.normal(18000, 9000, n_samples).clip(1000, 60000)
employment_years = rng.exponential(5, n_samples).clip(0, 40)
existing_debt = rng.normal(8000, 6000, n_samples).clip(0, 50000)
num_credit_lines = rng.integers(0, 12, n_samples)
has_defaulted_before = rng.choice([0, 1], size=n_samples, p=[0.88, 0.12])

debt_to_income = (existing_debt + loan_amount * 0.15) / annual_income
loan_to_income = loan_amount / annual_income

X_raw = pd.DataFrame({
    "annual_income": annual_income.round(0),
    "credit_score": credit_score.round(0),
    "loan_amount": loan_amount.round(0),
    "employment_years": employment_years.round(1),
    "existing_debt": existing_debt.round(0),
    "num_credit_lines": num_credit_lines,
    "has_defaulted_before": has_defaulted_before,
    "debt_to_income": debt_to_income.round(3),
    "loan_to_income": loan_to_income.round(3),
})

z = (
    0.018 * (credit_score - 650)
    - 3.0 * debt_to_income
    - 2.0 * loan_to_income
    + 0.10 * employment_years
    - 1.8 * has_defaulted_before
    + 0.00002 * (annual_income - 60000)
    - 0.05 * num_credit_lines
    + rng.normal(0, 1.0, n_samples)  # noise: real life isn't fully predictable
)
prob_approve = 1 / (1 + np.exp(-z))
approved = (rng.random(n_samples) < prob_approve).astype(int)

y = pd.Series(approved, name="approved")  # 1 = approved, 0 = denied

print("=" * 60)
print("STEP 1: SYNTHETIC DATASET GENERATED")
print("=" * 60)
print(f"Number of applications: {X_raw.shape[0]}")
print(f"Number of features: {X_raw.shape[1]}")
print(f"Approval rate: {y.mean():.1%}\n")

print("=" * 60)
print("STEP 2: A QUICK LOOK AT THE DATA")
print("=" * 60)
print(X_raw.head(), "\n")
print(X_raw.describe().T[["mean", "std", "min", "max"]], "\n")
print("Approved vs Denied counts:")
print(y.value_counts().rename({1: "approved", 0: "denied"}), "\n")



STEP 1: SYNTHETIC DATASET GENERATED
Number of applications: 2000
Number of features: 9
Approval rate: 39.2%

STEP 2: A QUICK LOOK AT THE DATA
   annual_income  credit_score  loan_amount  employment_years  existing_debt  \
0        66704.0         646.0      20279.0               3.4         2902.0   
1        37120.0         630.0      26057.0              14.0        14928.0   
2        76510.0         713.0      20460.0               4.9        11528.0   
3        80692.0         699.0      38149.0               8.8        14895.0   
4        17077.0         575.0      30868.0              28.2        17358.0   

   num_credit_lines  has_defaulted_before  debt_to_income  loan_to_income  
0                 0                     0           0.089           0.304  
1                 9                     0           0.507           0.702  
2                 1                     0           0.191           0.267  
3                11                     0           0.256           0.473